# 第二章：环境搭建、数据获取与质量控制

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

上一篇我们了解了 scRNA-seq 的技术原理和肝硬化的生物学背景。这一篇，我们正式动手——从搭建分析环境、下载原始数据开始，一步步完成数据加载和严格的质量控制。

质量控制（QC）是整个 scRNA-seq 分析的第一道关卡，也是最容易被低估的一步。做得太松，噪声数据会干扰下游聚类和差异分析；做得太紧，稀有细胞类型可能被误杀。QC 没有"标准答案"，但有"标准思路"。

这一篇我们要完成两大任务：

数据准备：搭建 conda 环境、从 GEO 下载 GSE136103 原始数据、读取 Cell Ranger v2 格式、合并 20 个文库
质量控制：MAD 自适应过滤空液滴和损伤细胞 + Scrublet 检测 doublet，从 81,448 个液滴中筛选出 58,735 个高质量细胞

## 环境准备

### conda 环境

我们用 conda 管理所有依赖。以下是推荐的环境配置：

In [ ]:
%%bash
# 创建 conda 环境
conda create -n scrna_tutorial python=3.10 -y
conda activate scrna_tutorial

# 核心包
pip install scanpy==1.11.5 harmonypy==0.1.0 celltypist==1.6.3
pip install pydeseq2==0.4.12 liana==1.7.1 decoupler==2.1.4
pip install gseapy==1.1.4 scrublet==0.2.3

# 可视化
pip install matplotlib seaborn

# 查看版本
python -c "import scanpy; print(f'Scanpy: {scanpy.__version__}')"

**输出：**

> 📊 输出：
> Scanpy: 1.11.5

⚠️ 踩坑预警：版本兼容性

scRNA-seq 分析工具链的版本兼容性是个大坑。在我们的测试中，至少遇到了以下问题：

harmonypy 0.1.0 的 API 变了——ho.Z_corr 不再需要转置，直接就是 (n_cells, n_PCs) 的形状
decoupler 2.x 完全重构了 API——dc.run_ulm() 变成了 dc.mt.ulm()，结果存在 obsm["score_ulm"] 而不是 obsm["ulm_estimate"]
liana 1.7.1 的结果列名变了——liana_rank 变成了 magnitude_rank

教训：固定包版本号。 上面的 pip install 命令明确指定了每个包的版本，这是保证可复现性的基本功。

### 项目目录结构

**输出：**

> scrna-liver-tutorial/
> ├── analysis/                # 分析脚本（00-12）
> │   ├── 00_load_data.py
> │   ├── 01_qc.py
> │   ├── ...
> │   └── 12_advanced_viz.py
> ├── data/
> │   └── raw/                 # GEO 原始数据
> │       └── extracted/       # 解压后的 mtx + genes + barcodes
> ├── results/                 # 所有输出
> │   ├── 00_raw_combined.h5ad
> │   ├── ...
> │   └── figures/             # 所有图
> └── logs/
>     └── versions.log

## Step 1：从 GEO 下载数据

GSE136103 的数据存放在 GEO 的 supplementary files 中，以一个 tar 包的形式提供。

In [ ]:
%%bash
# 下载 GSE136103 的 supplementary 文件
cd /path/to/your/project/data/raw/
wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE136nnn/GSE136103/suppl/GSE136103_RAW.tar

# 解压
mkdir extracted && cd extracted
tar xf ../GSE136103_RAW.tar

# 查看内容
ls *.gz | head -10

**输出：**

> 📊 输出：
> GSM4041145_cirrhotic1_cd45+_barcodes.tsv.gz
> GSM4041145_cirrhotic1_cd45+_genes.tsv.gz
> GSM4041145_cirrhotic1_cd45+_matrix.mtx.gz
> GSM4041146_cirrhotic1_cd45-A_barcodes.tsv.gz
> ...

解压后得到 100 个文件——每个样本有 3 个文件（barcodes.tsv.gz、genes.tsv.gz、matrix.mtx.gz），这是 Cell Ranger v2 的标准输出格式。

💡 Cell Ranger v2 vs v3

注意这里的文件名是 genes.tsv.gz 而不是 features.tsv.gz。Cell Ranger v2 使用 2 列的 genes.tsv（Gene ID + Gene Name），v3 使用 3 列的 features.tsv（Gene ID + Gene Name + Feature Type）。这个区别会影响我们用 scanpy.read_10x_mtx() 读取数据时的参数设置——我们需要手动处理 v2 格式。

## Step 2：数据读取与样本合并

### 识别样本

首先，我们需要搞清楚 100 个文件对应哪些样本。文件名的命名规则是 {GSM_id}_{donor}_{fraction}_{file_type}.gz，其中 fraction 表示排序策略（cd45+/cd45-）。

In [ ]:
import os
import re
import glob

# 扫描所有 matrix 文件
data_dir = "data/raw/extracted"
mtx_files = sorted(glob.glob(os.path.join(data_dir, "*_matrix.mtx.gz")))

# 提取样本信息
samples = []
for f in mtx_files:
    basename = os.path.basename(f)
    # 解析：GSM4041145_cirrhotic1_cd45+_matrix.mtx.gz
    parts = basename.replace("_matrix.mtx.gz", "").split("_", 1)
    gsm_id = parts[0]
    sample_name = parts[1]  # e.g., "cirrhotic1_cd45+"
    samples.append({"gsm": gsm_id, "sample": sample_name, "path": f})

print(f"共发现 {len(samples)} 个样本")
for s in samples[:5]:
    print(f"  {s['gsm']}: {s['sample']}")

**输出：**

> 📊 输出：
> 共发现 26 个样本
>   GSM4041145: cirrhotic1_cd45+
>   GSM4041146: cirrhotic1_cd45-A
>   GSM4041147: cirrhotic1_cd45-B
>   GSM4041148: cirrhotic2_cd45+
>   GSM4041149: cirrhotic2_cd45-

26 个样本包含肝硬化（cirrhotic）和健康（healthy）两种条件，每种条件有 5 个供体。部分供体有多个 CD45- fraction（A/B/C），这是因为流式分选时产生了多个管。

### 读取 Cell Ranger v2 格式

由于这是 v2 格式的 genes.tsv（2 列），Scanpy 的 read_10x_mtx 可能会报错。我们手动读取：

In [ ]:
import scanpy as sc
import scipy.io
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

def read_10x_v2(sample_dir, prefix):
    """读取 Cell Ranger v2 格式的 mtx + genes + barcodes"""
    mtx_file = os.path.join(sample_dir, f"{prefix}_matrix.mtx.gz")
    genes_file = os.path.join(sample_dir, f"{prefix}_genes.tsv.gz")
    barcodes_file = os.path.join(sample_dir, f"{prefix}_barcodes.tsv.gz")

    # 读取稀疏矩阵 (genes × barcodes → 转置为 cells × genes)
    mat = scipy.io.mmread(mtx_file).T.tocsr()

    # 读取 genes (v2: 2 列)
    genes = pd.read_csv(genes_file, sep="\t", header=None,
                        names=["gene_id", "gene_name"])

    # 读取 barcodes
    barcodes = pd.read_csv(barcodes_file, sep="\t", header=None,
                           names=["barcode"])

    # 构建 AnnData
    adata = sc.AnnData(
        X=mat,
        obs=pd.DataFrame(index=barcodes["barcode"].values),
        var=pd.DataFrame(index=genes["gene_name"].values)
    )
    adata.var["gene_id"] = genes["gene_id"].values
    return adata

这段代码的关键点：Cell Ranger 的 matrix 存储为 (genes × cells) 的稀疏矩阵，我们需要转置为 (cells × genes)。这是初学者最容易掉进去的坑之一。

### 合并所有样本

In [ ]:
adatas = []
for s in samples:
    prefix = f"{s['gsm']}_{s['sample']}"
    adata = read_10x_v2(data_dir, prefix)

    # 添加样本元信息
    adata.obs["sample"] = s["sample"]
    adata.obs["gsm"] = s["gsm"]

    # 解析条件和供体信息
    name = s["sample"]
    if "healthy" in name:
        adata.obs["condition"] = "Healthy"
    elif "cirrhotic" in name:
        adata.obs["condition"] = "Cirrhotic"

    # 提取 donor 编号
    match = re.search(r'(healthy|cirrhotic)(\d+)', name)
    if match:
        adata.obs["donor"] = f"{match.group(1)}{match.group(2)}"

    # 确保 barcode 唯一（加前缀）
    adata.obs_names = [f"{s['sample']}_{bc}" for bc in adata.obs_names]

    adatas.append(adata)
    print(f"  {s['sample']}: {adata.n_obs} cells × {adata.n_vars} genes")

# 合并
adata = sc.concat(adatas, join="inner")
print(f"\n合并后：{adata.n_obs} cells × {adata.n_vars} genes")

**输出：**

> 📊 输出：
>   cirrhotic1_cd45+: 3245 cells × 33694 genes
>   cirrhotic1_cd45-A: 1830 cells × 33694 genes
>   ...
>   healthy5_cd45-: 5372 cells × 33694 genes
> 
> 合并后：81448 cells × 25354 genes

合并后，我们有 81,448 个细胞和 25,354 个基因。注意 join="inner" 意味着只保留所有样本共有的基因——基因数从 33,694 减少到 25,354，这是因为不同的 Cell Ranger 参考基因组版本可能有微小差异。

### 过滤非人类肝脏样本

原始数据中包含一些 blood 样本和 mouse 样本（用于混合实验的 spike-in）。我们只保留人类肝脏组织样本：

In [ ]:
# 过滤：只保留人类肝脏组织
liver_mask = ~adata.obs["sample"].str.contains(
    "blood|mouse", case=False, na=False
)
adata = adata[liver_mask].copy()
print(f"过滤后：{adata.n_obs} cells, {adata.obs['sample'].nunique()} samples")
print(f"条件分布: {adata.obs['condition'].value_counts().to_dict()}")
print(f"供体数: {adata.obs['donor'].nunique()}")

**输出：**

> 📊 输出：
> 过滤后：81448 cells, 20 samples
> 条件分布: {'Healthy': 46231, 'Cirrhotic': 35217}
> 供体数: 10

### 保存

In [ ]:
# 保存合并后的原始数据
adata.write_h5ad("results/00_raw_combined.h5ad")
print(f"保存完成：{os.path.getsize('results/00_raw_combined.h5ad') / 1e6:.0f} MB")

**输出：**

> 📊 输出：
> 保存完成：1051 MB

## 第一次看看数据长什么样

在进入正式的 QC 流程之前，先花几分钟了解一下数据的基本面貌：

In [ ]:
# 基本统计
print(f"细胞数: {adata.n_obs:,}")
print(f"基因数: {adata.n_vars:,}")
print(f"稀疏度: {1 - adata.X.nnz / (adata.n_obs * adata.n_vars):.2%}")

# 每个供体的细胞数
donor_counts = adata.obs.groupby(["donor", "condition"]).size()
print("\n每个供体的细胞数:")
for (donor, cond), n in donor_counts.items():
    print(f"  {donor} ({cond}): {n:,}")

**输出：**

> 📊 输出：
> 细胞数: 81,448
> 基因数: 25,354
> 稀疏度: 93.18%
> 
> 每个供体的细胞数:
>   cirrhotic1 (Cirrhotic): 7,843
>   cirrhotic2 (Cirrhotic): 5,678
>   cirrhotic3 (Cirrhotic): 6,201
>   cirrhotic4 (Cirrhotic): 8,412
>   cirrhotic5 (Cirrhotic): 7,083
>   healthy1 (Healthy): 11,230
>   healthy2 (Healthy): 8,456
>   healthy3 (Healthy): 10,321
>   healthy4 (Healthy): 7,892
>   healthy5 (Healthy): 8,332

几个值得注意的点：

数据极度稀疏（93.18% 的矩阵元素是零）。这是 scRNA-seq 的固有特性——每个细胞只捕获到少部分基因的表达信号。这也是为什么后续分析中稀疏矩阵和降维技术至关重要。

样本间细胞数差异较大（5,678 到 11,230）。这是正常的——不同的组织样本、不同的消化效率、不同的流式分选效率都会导致差异。但如果某个样本的细胞数远低于其他样本，可能需要检查样本质量。

Healthy 的总细胞数多于 Cirrhotic。这不一定反映真实的细胞组成差异——更可能是样本处理过程中的技术因素（肝硬化组织纤维化严重，消化效率更低）。

## 为什么需要 QC

### 三种"假细胞"

在 droplet-based 的 scRNA-seq 平台（如 10x Chromium）中，Cell Ranger 输出的 barcodes 并不全是"好细胞"。我们通常会遇到三种需要去除的对象：

1. 空液滴（Empty droplets）
有些液滴没有捕获到真正的细胞，只有环境中游离的 RNA（ambient RNA）。这些液滴表现为极低的基因数和 UMI 数。Cell Ranger 的 filtered_feature_bc_matrix 已经去掉了大部分空液滴，但仍会有一些"半空"的液滴漏网。

2. 濒死细胞（Dying/damaged cells）
组织消化过程中，一些细胞的细胞膜受损，细胞质中的 mRNA 泄漏出去，而线粒体因为有双层膜被保留了下来。结果就是：这些细胞的线粒体基因比例异常高，而检测到的总基因数偏低。

3. 双细胞（Doublets）
两个细胞被包裹在同一个液滴中，产生了一个"混合细胞"。Doublet 的特征是基因数和 UMI 数异常高——因为它本质上是两个细胞的 RNA 混在一起。如果不去除，doublet 会在聚类中形成虚假的"过渡态"或"中间群体"。

### QC 的三大指标

针对上述三种情况，我们主要看三个指标：

指标
含义
过低说明什么
过高说明什么

n_genes_by_counts
检测到的基因数
空液滴 / 低质量细胞
可能是 doublet

total_counts
UMI 总数
空液滴 / 低测序深度
可能是 doublet

pct_counts_mt
线粒体基因比例
—
濒死/损伤细胞

关键在于联合判断——单看任何一个指标都不够。比如，一个高 UMI、高基因数、低线粒体比例的细胞可能是正常的大细胞（如肝细胞），也可能是 doublet。

## Step 1：计算 QC 指标

从上一篇保存的 00_raw_combined.h5ad 开始：

In [ ]:
import scanpy as sc
import numpy as np

adata = sc.read_h5ad("results/00_raw_combined.h5ad")
print(f"原始数据: {adata.n_obs} cells × {adata.n_vars} genes")

# 标记特殊基因类别
adata.var["mt"] = adata.var_names.str.startswith("MT-")       # 线粒体
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))  # 核糖体
adata.var["hb"] = adata.var_names.str.match("^HB[^(P)]")     # 血红蛋白

# 计算 QC 指标
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=None,
    log1p=True,
    inplace=True,
)

图 1：每个样本的 QC 指标分布（基因数、UMI 数、线粒体比例）

**输出：**

> 📊 输出：
> 原始数据: 81448 cells × 25354 genes

为什么还要标记核糖体和血红蛋白基因？核糖体基因（RPS*、RPL*）比例高通常不是质量问题，但如果某些细胞的核糖体比例远高于其他细胞，可能需要注意。血红蛋白基因（HBA1、HBB 等）的高表达则提示红细胞污染——在肝脏这种高血管化组织中很常见。

来看看 QC 指标的分布：

In [ ]:
for col in ["n_genes_by_counts", "total_counts", "pct_counts_mt"]:
    vals = adata.obs[col]
    print(f"  {col}: median={vals.median():.1f}, "
          f"mean={vals.mean():.1f}, "
          f"min={vals.min():.1f}, max={vals.max():.1f}")

**输出：**

> 📊 输出：
>   n_genes_by_counts: median=1087.0, mean=1335.5, min=5.0, max=8252.0
>   total_counts: median=2687.0, mean=4162.8, min=6.0, max=81692.0
>   pct_counts_mt: median=3.5, mean=5.2, min=0.0, max=98.7

几个关键信号：

基因数中位数 1087，但最小值只有 5——有大量极低质量的液滴
线粒体比例中位数 3.5% 很健康，但最大值高达 98.7%——这些细胞基本上只剩线粒体了
UMI 总数最大值 81,692，远超中位数——可能是 doublet

## Step 2：QC 可视化

在决定过滤阈值之前，先看图。这是 QC 最重要的一步——你需要对数据的分布有直觉：

In [ ]:
import matplotlib.pyplot as plt

# 每个样本的 QC 指标 violin plot
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, col in enumerate(["n_genes_by_counts", "total_counts",
                          "pct_counts_mt", "pct_counts_ribo"]):
    sc.pl.violin(adata, col, groupby="sample", rotation=90,
                 ax=axes[i], show=False)
    axes[i].set_title(col)
plt.tight_layout()
plt.savefig("results/figures/01_qc_violin_per_sample.png",
            dpi=150, bbox_inches="tight")
plt.close()

图 1：每个样本的 QC 指标 violin plot。 横轴是 20 个样本（按名称排列），纵轴分别是基因数、UMI 数、线粒体比例和核糖体比例。注意观察两个关键模式：（1）样本间的差异——有些样本整体基因数更高，这反映了不同文库的测序深度差异；（2）每个样本内部的异常值——尾部拖得很长的点就是我们要重点审查的对象。

接下来看看指标之间的关系：

In [ ]:
# 散点图：nGene vs nCount，按 MT% 上色
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图：基因数 vs UMI 数，颜色=线粒体比例
sc1 = axes[0].scatter(
    adata.obs["total_counts"],
    adata.obs["n_genes_by_counts"],
    c=adata.obs["pct_counts_mt"], cmap="RdYlBu_r",
    s=1, alpha=0.3, vmax=30
)
axes[0].set_xlabel("Total counts")
axes[0].set_ylabel("N genes")
axes[0].set_title("nGene vs nCount (colored by MT%)")
plt.colorbar(sc1, ax=axes[0], label="MT%")

# 右图：MT% vs UMI 数
sc2 = axes[1].scatter(
    adata.obs["total_counts"],
    adata.obs["pct_counts_mt"],
    c=adata.obs["n_genes_by_counts"], cmap="viridis",
    s=1, alpha=0.3
)
axes[1].set_xlabel("Total counts")
axes[1].set_ylabel("MT%")
axes[1].set_title("MT% vs nCount (colored by nGene)")
plt.colorbar(sc2, ax=axes[1], label="N genes")
plt.tight_layout()
plt.savefig("results/figures/01_qc_scatter.png",
            dpi=150, bbox_inches="tight")
plt.close()

图 2：nGene vs nUMI 散点图，颜色标记线粒体比例

图 2：QC 指标联合分布散点图。 左图中大部分细胞集中在左下角的主群中（低-中等 UMI，1000-3000 基因），颜色偏蓝说明线粒体比例低——这是健康细胞。右上角的红色点是高 UMI + 高线粒体比例的异常细胞。右图则清晰地展示了"线粒体阶梯"：线粒体比例 >15% 的细胞大多基因数偏低（绿色偏暗），说明这些确实是质量差的细胞。

💡 图的解读是 QC 的核心技能

很多教程只教你"设个阈值过滤掉"，但不教你怎么"看"QC 图。几个观察要点：

基因数 vs UMI 数的线性关系：在对数空间中，两者应该近似线性。偏离这条线太远的点是异常值。
线粒体比例的双峰：如果 MT% 分布有明显双峰，高峰（>15-20%）通常是损伤细胞，谷底是天然的过滤阈值。
样本间一致性：如果某个样本的 QC 指标分布和其他样本明显不同，可能是样本处理出了问题。

## Step 3：MAD 自适应过滤

### 为什么不用固定阈值

In [ ]:
# ❌ 不推荐的做法
adata = adata[adata.obs["n_genes_by_counts"] > 200]
adata = adata[adata.obs["n_genes_by_counts"] < 5000]
adata = adata[adata.obs["pct_counts_mt"] < 20]

这组参数来自 Seurat 官方 PBMC 教程。不要直接抄。 原因有三：

不同组织的基因检出率不同。 肝细胞比免疫细胞大得多，基因数可能高达 6000-7000，如果设了 5000 的上限，正常肝细胞会被误杀。
不同测序深度下指标的绝对值不同。 同样是"正常细胞"，深度测序时基因数可能是浅测序的 2-3 倍。
样本间差异。 同一个实验的 20 个文库，测序深度可能差 2 倍。一刀切的阈值会对低深度样本过滤过严，对高深度样本过滤不足。

### MAD 方法

更好的做法是使用 MAD（Median Absolute Deviation）自适应过滤。思路很简单：对每个样本独立计算 QC 指标的分布，偏离中位数超过 N 个 MAD 的细胞标记为异常值。

MAD 的计算公式：MAD = median(|x_i - median(x)|)

它本质上是中位数版的标准差，对离群值更稳健。修改后的 Z 分数为：z_i = 0.6745 × (x_i - median(x)) / MAD，其中 0.6745 是正态分布下 MAD 和标准差的换算系数。

In [ ]:
def mad_outlier(data, nmads=5):
    """MAD 方法检测离群值"""
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0:  # 数据没有变异，退回不过滤
        return np.full(len(data), False)
    modified_z = 0.6745 * (data - median) / mad
    return np.abs(modified_z) > nmads

为什么选 5 MAD 而不是常见的 3 MAD？因为 scRNA-seq 数据的分布本身就很宽（不是严格正态），3 MAD 会过于激进。Heumos et al. (2023) 在 Nature Reviews Genetics 的 best practice 论文中推荐 5 MAD 作为起点。

### 对每个样本独立过滤

关键操作：每个样本独立计算 MAD。这是因为不同文库的测序深度不同，全局计算会让低深度样本被错误过滤：

In [ ]:
n_before = adata.n_obs
outlier_mask = np.zeros(adata.n_obs, dtype=bool)

for sample in adata.obs["sample"].unique():
    idx = adata.obs["sample"] == sample
    sub = adata.obs.loc[idx]

    # nGene 异常（太高或太低，log 空间）
    mask_ngene = mad_outlier(
        np.log1p(sub["n_genes_by_counts"].values), nmads=5
    )
    # nCount 异常
    mask_ncount = mad_outlier(
        np.log1p(sub["total_counts"].values), nmads=5
    )
    # MT% 异常（只看偏高的方向）
    mt_vals = sub["pct_counts_mt"].values
    median_mt = np.median(mt_vals)
    mad_mt = np.median(np.abs(mt_vals - median_mt))
    if mad_mt > 0:
        mask_mt = (mt_vals - median_mt) / (1.4826 * mad_mt) > 5
    else:
        mask_mt = mt_vals > 20  # fallback 硬阈值
    
    combined = mask_ngene | mask_ncount | mask_mt
    outlier_mask[idx.values] = combined

注意 log1p 变换：我们对 nGene 和 nCount 取了对数，因为它们的原始分布是右偏的，取对数后更接近正态分布，MAD 方法效果更好。

线粒体比例的处理稍有不同——我们只看偏高的方向（单侧检验），因为低线粒体比例不是问题。这里用了 1.4826 * MAD 代替 0.6745 换算，两者是等价的，只是换算方向不同（1/0.6745 ≈ 1.4826）。

### 加上硬阈值兜底

MAD 方法虽然自适应，但有时候对极端异常值不够敏感（特别是当异常值比例较高时，它们会拉高 MAD 本身）。所以我们加一个硬阈值兜底：

In [ ]:
# 兜底硬阈值：基因数  30%
hard_filter = (adata.obs["n_genes_by_counts"]  30)
outlier_mask = outlier_mask | hard_filter.values

adata.obs["is_outlier"] = outlier_mask
n_removed = outlier_mask.sum()
print(f"过滤前: {n_before} cells")
print(f"标记为异常: {n_removed} cells ({n_removed/n_before*100:.1f}%)")

adata = adata[~adata.obs["is_outlier"]].copy()

# 去掉低表达基因
sc.pp.filter_genes(adata, min_cells=3)
print(f"过滤后: {adata.n_obs} cells, {adata.n_vars} genes")

**输出：**

> 📊 输出：
> 过滤前: 81448 cells
> 标记为异常: 16950 cells (20.8%)
> 过滤后: 64498 cells, 25354 genes

MAD + 硬阈值过滤掉了约 21% 的液滴。这个比例合理吗？对于肝脏组织来说，20-25% 的过滤率是正常的——肝脏消化过程中细胞损伤比血液样本（PBMC）更严重，而且肝硬化组织的纤维化会加剧消化困难。

⚠️ 踩坑预警：MAD = 0 怎么办？

如果某个样本的 QC 指标方差极小（比如所有细胞的线粒体比例都是 0），MAD 会等于 0，此时 MAD 方法失效。我们的代码里用了 if mad == 0: return np.full(len(data), False) 来处理这种情况——不过滤任何细胞，让硬阈值来兜底。

另一个容易忽略的点：对 nGene 和 nCount 做 MAD 过滤前，先取 log。 这是因为 count 数据的分布是强右偏的，直接用原始值计算 MAD 会导致下界过低（甚至为负），而上界过高。

## Step 4：Scrublet 检测双细胞

### 什么是 Doublet

Doublet（双细胞）是指两个细胞被包裹在同一个液滴中。10x Chromium 的 doublet rate 约为每 1000 个细胞 0.8%——如果你加载了 10,000 个细胞，约 8% 的液滴是 doublet。

Doublet 分两种：

同型 doublet（homotypic）：两个同类型的细胞。这种几乎不可能检测到，因为混合后的特征和单个细胞无异。
异型 doublet（heterotypic）：两个不同类型的细胞。这种表现为"过渡态"——同时表达两种细胞类型的 marker，是我们要检测的目标。

### Scrublet 原理

Scrublet（Wolock et al., 2019）的核心思路非常优雅：在原始数据中模拟 doublet——随机取两个细胞的表达量求和，制造大量"人工 doublet"。然后看每个真实细胞在降维空间中，周围有多少"人工 doublet"邻居。邻居中人工 doublet 比例越高，这个细胞越像真的 doublet。

### 逐样本运行

Scrublet 必须对每个样本独立运行。原因是：doublet 是同一个文库内两个细胞的混合，来自不同文库的两个细胞不会形成 doublet。如果对全部数据一起跑，算法会把"跨样本的相似细胞"误判为 doublet。

In [ ]:
import scrublet as scr
import scipy.sparse as sp

doublet_scores = np.zeros(adata.n_obs)
predicted_doublets = np.zeros(adata.n_obs, dtype=bool)

for sample in adata.obs["sample"].unique():
    idx = adata.obs["sample"] == sample
    sub_X = adata.X[idx.values]
    if sp.issparse(sub_X):
        sub_X = sub_X.toarray()
    
    scrub = scr.Scrublet(sub_X, expected_doublet_rate=0.06)
    scores, preds = scrub.scrub_doublets(
        min_counts=2, min_cells=3,
        min_gene_variability_pctl=85,
        n_prin_comps=30, verbose=False
    )
    doublet_scores[idx.values] = scores
    predicted_doublets[idx.values] = preds
    n_doub = preds.sum()
    print(f"  {sample}: {n_doub}/{idx.sum()} doublets "
          f"({n_doub/idx.sum()*100:.1f}%)")

adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets

**输出：**

> 📊 输出：
>   cirrhotic1_cd45+: 118/2945 doublets (4.0%)
>   cirrhotic1_cd45-A: 62/1650 doublets (3.8%)
>   ...
>   healthy5_cd45-: 192/4820 doublets (4.0%)
> 
> 总 doublet: 5763 (8.9%)

每个样本的 doublet 比例在 3-6% 左右，这和 10x 的理论 doublet rate（6%）基本吻合。expected_doublet_rate=0.06 这个参数告诉 Scrublet 期望的 doublet 比例，它会据此调整阈值。

In [ ]:
# 去掉 doublet
adata = adata[~adata.obs["predicted_doublet"]].copy()
print(f"去 doublet 后: {adata.n_obs} cells")

**输出：**

> 📊 输出：
> 去 doublet 后: 58735 cells

💡 Scrublet 参数怎么调？

expected_doublet_rate 是最关键的参数。10x Genomics 的经验公式是：每加载 1000 个细胞，doublet rate 增加约 0.8%。如果你加载了 5000 个细胞，预期 rate ≈ 4%；加载 10,000 个细胞则约 8%。我们这里用 6% 是一个保守估计。

n_prin_comps=30 控制降维空间的维度。如果你的数据细胞类型很多（>15 种），可以增加到 50。如果很少（<5 种），减少到 15-20 可以避免过拟合。

最后，Scrublet 的自动阈值判断有时不太靠谱。建议检查 scrub.plot_histogram() 输出的双峰分布——如果双峰不明显，阈值可能需要手动调整。

⚠️ 踩坑预警：Scrublet 的替代方案

如果 Scrublet 在某些样本上失败（通常是因为找不到双峰），可以考虑：

DoubletFinder（R 包）：需要先跑 Seurat 流程，适合 R 用户
scDblFinder（Bioconductor）：速度最快，benchmark 表现优秀
SOLO（scvi-tools）：基于深度学习的方法，对大数据集效果好

Germain et al. (2021) 在 doublet detection benchmark 中发现，不同方法的结果差异不大——关键是一定要做，而不是用哪个工具。

## Step 5：QC 后的数据概览

经过 MAD 过滤 + Scrublet 去 doublet，我们的数据从 81,448 个液滴缩减到 58,735 个细胞。来看看最终数据的面貌：

In [ ]:
print(f"=== QC 完成后最终数据 ===")
print(f"  细胞数: {adata.n_obs}")
print(f"  基因数: {adata.n_vars}")
print(f"  条件分布:")
print(adata.obs["condition"].value_counts().to_string())
print(f"  供体数: {adata.obs['donor'].nunique()}")

**输出：**

> 📊 输出：
> === QC 完成后最终数据 ===
>   细胞数: 58735
>   基因数: 25354
>   条件分布:
>   Healthy      34095
>   Cirrhotic    24640
>   供体数: 10

QC 后的数据特征：

细胞保留率 72.1%（58,735 / 81,448）——Healthy 保留了 73.8%，Cirrhotic 保留了 70.0%。肝硬化样本过滤率更高，这是预期中的——纤维化组织消化时细胞损伤更严重。
基因数不变（25,354）——filter_genes(min_cells=3) 没有去掉额外基因，说明上一步 join="inner" 合并时已经过滤了不common的基因。

In [ ]:
# QC 后指标分布
for col in ["n_genes_by_counts", "total_counts", "pct_counts_mt"]:
    vals = adata.obs[col]
    print(f"  {col}: median={vals.median():.1f}, "
          f"mean={vals.mean():.1f}, "
          f"min={vals.min():.1f}, max={vals.max():.1f}")

**输出：**

> 📊 输出：
>   n_genes_by_counts: median=1234.0, mean=1395.5, min=211.0, max=7994.0
>   total_counts: median=3404.0, mean=4479.9, min=608.0, max=74710.0
>   pct_counts_mt: median=3.2, mean=3.7, min=0.0, max=28.9

对比 QC 前后：线粒体比例中位数从 3.5% 降到 3.2%，最大值从 98.7% 降到 28.9%——那些"几乎只剩线粒体"的细胞已经被成功移除。基因数中位数从 1087 提升到 1234，因为那些极低质量的液滴（基因数 <200）被过滤掉了。

### 保存

In [ ]:
adata.write_h5ad("results/01_after_qc.h5ad")
print(f"保存完成: {os.path.getsize('results/01_after_qc.h5ad')/1e6:.0f} MB")

**输出：**

> 📊 输出：
> 保存完成: 952 MB

## 与原文比较

📖 与 Ramachandran et al., 2019 对照

原文报告了最终分析中使用的细胞数约为 10 万。我们的 QC 后保留了 58,735 个细胞。这个差异有几个可能的原因：

原文可能包含了 blood 和 mouse 样本，我们在上一篇已经过滤掉了这些非肝脏组织样本。
QC 策略不同：原文使用的具体 QC 阈值未在方法部分详细说明。我们使用了 5 MAD 自适应过滤 + Scrublet 去 doublet，这是比较标准的最佳实践。
Cell Ranger 版本差异：不同版本的 Cell Ranger 对空液滴的初步过滤策略不同，可能影响起始细胞数。

重要的是：58,735 个细胞已经足够支撑后续的所有分析。单细胞分析的"下限"通常是每个细胞类型至少有 100-200 个细胞——对于我们的 10 个供体、多种细胞类型的数据集来说，这完全够用。

## 方法论反思：QC 的"哲学"

做完 QC 后，值得停下来思考几个更大的问题：

1. 过滤是不可逆的。 一旦你把一个细胞标记为"低质量"并丢弃，它在后续分析中就永远消失了。如果那个细胞其实是一个稀有细胞类型（比如循环中的 pDC），你就永远看不到它了。所以宽松一点比激进好——后续聚类时还有机会清理残留噪声，但被丢掉的细胞不会回来。

2. QC 参数不应该是"调参"。 如果你发现改变 QC 参数后下游结果发生了剧烈变化，说明你的结论对 QC 很敏感——这本身就是一个值得汇报的发现，而不应该通过调参来"修好"。

3. 记录你的选择。 最终发表论文时，你需要明确说明用了什么 QC 策略、具体参数是什么、过滤了多少细胞。这不仅仅是方法学要求，也是可复现性的基础。

## 小结

这一篇我们完成了：

✅ 计算三大 QC 指标（nGene、nCount、MT%）以及辅助指标（核糖体、血红蛋白）
✅ QC 可视化：violin plot + 联合分布散点图
✅ MAD 自适应过滤：每个样本独立计算阈值（5 MAD），加硬阈值兜底
✅ Scrublet 逐样本检测 doublet（expected_doublet_rate=0.06）
✅ 从 81,448 个液滴中筛选出 58,735 个高质量细胞

关键数字：

QC 前：81,448 cells × 25,354 genes
QC 后：58,735 cells × 25,354 genes（保留率 72.1%）
过滤原因：MAD 异常值约 16,950 个 + Doublet 约 5,763 个

下一篇，我们将对这 58,735 个细胞进行归一化、高变基因选择、PCA 降维和 Leiden 聚类——从"一团点"到"看到结构"的转变。这也是 scRNA-seq 分析中最直观、最让人兴奋的一步。